# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Naveed-Qasim608/Flyrank_ML_Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of analysis + time window

The unit of analysis is **one content page per client over a defined time window**.

Each row represents one content item with its observed search performance signals.

The analysis uses historical search performance windows, such as the previous 30–90 days, to compare recent performance changes.

The goal is to identify directional signals related to content decline and prioritize pages for review.

In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Fields

### Features
Signals used by the model:
- impressions_90d
- avg_position
- ctr
- content_age_days
- days_since_last_update
- word_count

### Label
Target variable:
- is_declining

This label is created from observed trend_direction:
- down = declining
- otherwise = not declining

### Context
Additional information used for understanding:
- client_hash_id
- content_hash_id
- report_date
- trend_direction

### Excluded
Fields excluded from modeling:
- Any identifiers that do not describe page performance
- Future outcome fields
- Any decision flags created after analysis

Reason:
These fields could introduce leakage or do not represent information available before making a refresh decision.

In [ ]:
# Check which planned fields exist in the dataset

planned_fields = [
    "impressions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "word_count",
    "trend_direction",
    "client_hash_id",
    "content_hash_id"
]

available_fields = [
    col for col in planned_fields
    if col in df.columns
]

print("Available fields:")
print(available_fields)

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## Data verification

The following checks verify:
- dataframe size
- available fields
- missing values
- time coverage

These checks confirm that the data contract is based on measured observations rather than assumptions.

In [ ]:
# Dataset size

print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))


# Missing value check

missing = df.isnull().sum()

print("\nMissing values:")
print(missing.sort_values(ascending=False).head(10))


# Trend label distribution

df["is_declining"] = (
    df["trend_direction"]
    .str.lower()
    .eq("down")
    .astype(int)
)

print("\nLabel distribution:")
print(df["is_declining"].value_counts())

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data limits

This dataset has several limitations:

- It cannot prove why a page declined. It only shows observed relationships between signals and performance changes.
- Client history is unbalanced, meaning some clients have longer historical data than others.
- Early rows may contain limited Google Search Console history.
- Overlapping time windows may cause similar observations to appear multiple times.

The dataset supports directional analysis and decision-support, but it cannot provide causal explanations or predict Google's ranking algorithm.

In [ ]:
# Check date range if available

date_columns = [
    col for col in df.columns
    if "date" in col.lower()
]

print("Date columns found:")
print(date_columns)

if len(date_columns) > 0:
    for col in date_columns:
        print("\n", col)
        print("Min:", df[col].min())
        print("Max:", df[col].max())

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.